In [1]:
import pandas as pd
import numpy as np
import kmapper as km
from sklearn import ensemble, cluster
import pandas as pd
import dash
from dash import dcc, html, Dash, Input, Output, dash_table
import plotly.graph_objects as go
import math
import dash_cytoscape as cyto
from thresholds import thresholds

dataset_path = '../datasets/db_nl_preprocessed-edit.csv'

df = pd.read_csv(dataset_path)
df[np.isnan(df)] = 0
features = [c for c in df.columns]
X = np.array(df[features])

projector = ensemble.IsolationForest(random_state=0, n_jobs=-1)
projector.fit(X)
lens1 = projector.decision_function(X)

mapper = km.KeplerMapper(verbose=3)
lens2 = mapper.fit_transform(X, projection="knn_distance_5")
lens = np.c_[lens1, lens2]

G = mapper.map(
    lens,
    X,
    cover = km.Cover(n_cubes=20,
                     perc_overlap=.7),
    clusterer=cluster.AgglomerativeClustering(3))

print(f"num nodes: {len(G['nodes'])}")
print(f"num edges: {sum([len(values) for key, values in G['links'].items()])}")

_ = mapper.visualize(G, path_html="../results/kepler-mapper.html")

app = dash.Dash(__name__)

node_ids = list(G['nodes'].keys())
node_counts = [len(members) for members in G['nodes'].values()]

fig = go.Figure(data=go.Bar(x=node_ids, y=node_counts, marker_color='blue'))
fig.update_layout(title='Количество строк на ноду в графе Mapper', xaxis_title='Node ID', yaxis_title='Количество строк')

app.layout = html.Div([
    html.H1('Интерактивный дашборд для анализа графа Mapper'),
    dcc.Graph(figure=fig),
])

if __name__ == '__main__':
    app.run_server(debug=True, port=8061)

# Расчет логарифмического размера каждой узлы на основе данных
node_sizes = {}
for index, members in G['nodes'].items():
    count = len(set(members))
    node_sizes[index] = count

# Минимальный и максимальный размеры узлов
min_size = 2  # Минимальный размер узла
max_size = 9  # Максимальный размер узла

# Находим минимальное и максимальное значение для корректного масштабирования
min_count = min(node_sizes.values()) if node_sizes else 1
max_count = max(node_sizes.values()) if node_sizes else 1

# Нормализация размеров с использованием логарифмической шкалы
normalized_sizes = {}
for node, count in node_sizes.items():
    if count > 1:
        # Применяем логарифмическое масштабирование
        log_size = math.log(count, max_count)  # База логарифма — максимальное значение count
        normalized_size = min_size + (max_size - min_size) * (log_size / math.log(max_count, max_count))
    else:
        normalized_size = min_size
    normalized_sizes[node] = normalized_size

# Обновляем стиль узлов
elements = [
    {'data': {'id': node, 'label': node}, 'style': {'width': normalized_sizes[node], 'height': normalized_sizes[node]}}
    for node in G['nodes']]

elements += [
    {'data': {'source': edge[0], 'target': edge[1]}}
    for edge in G['simplices'] if isinstance(edge, (list, tuple)) and len(edge) >= 2
]

stylesheet = [
    {
        'selector': 'node',
        'style': {
            'background-color': 'gray',
            'label': 'data(label)',
            'font-size': '1px',
        }
    },
    {
        'selector': 'node:selected',
        'style': {'background-color': 'red'}
    },
    {
        'selector': 'edge',
        'style': {
            'line-color': 'light-gray',
            'width': 0.1
        }
    }
]

# Функция для создания условных стилей для DataTable
def generate_style_data_conditional(df, thresholds):
    styles = []
    for column, bounds in thresholds.items():
        # Ниже нормы (светло-синий)
        styles.append({
            'if': {
                'filter_query': f'{{{column}}} < {bounds["low"]}',
                'column_id': column
            },
            'backgroundColor': '#add8e6',
            'color': 'black'
        })
        # В норме (светло-зелёный)
        styles.append({
            'if': {
                'filter_query': f'{{{column}}} >= {bounds["low"]} && {{{column}}} <= {bounds["high"]}',
                'column_id': column
            },
            'backgroundColor': '#90ee90',
            'color': 'black'
        })
        # Выше нормы (светло-красный)
        styles.append({
            'if': {
                'filter_query': f'{{{column}}} > {bounds["high"]}',
                'column_id': column
            },
            'backgroundColor': '#ffcccb',
            'color': 'black'
        })
        # Пустой
        styles.append({
            'if': {
                'filter_query': f'{{{column}}} is blank',
                'column_id': column
            },
            'backgroundColor': '#ffffff',
            'color': 'black'
        })
    return styles


conditional_styles = generate_style_data_conditional(df, thresholds)

app = Dash(__name__)
server = app.server
app.layout = html.Div([
    cyto.Cytoscape(
        id='cytoscape-graph',
        elements=elements,
        stylesheet=stylesheet,
        layout={'name': 'cose'},
        style={'width': '100%', 'height': '900px'},
        boxSelectionEnabled=True
    ),
    html.Div(id='selected-node-data', style={'padding-top': '20px'}),
])

@app.callback(
    Output('selected-node-data', 'children'),
    [Input('cytoscape-graph', 'selectedNodeData')]
)
def update_output(selected_nodes):
    if not selected_nodes:
        return "No nodes selected"
    else:
        # Получаем список выбранных идентификаторов узлов
        node_ids = [node['id'] for node in selected_nodes]
        all_rows_list = []
        for node_id in node_ids:
            # Используем напрямую G['nodes'] для получения индексов строк
            rows_list = G['nodes'].get(node_id, [])
            all_rows_list.extend(rows_list)
        unique_rows_list = list(set(all_rows_list))
        selected_rows = df.iloc[unique_rows_list]

        return dash_table.DataTable(
            data=selected_rows.to_dict('records'),
            columns=[{"name": i, "id": i} for i in selected_rows.columns],
            page_size=100,
            style_table={'height': '1000px', 'overflowY': 'auto'},
            fixed_rows={'headers': True, 'data': 0},
            style_data_conditional=conditional_styles,
        )

/Users/umanindaniil/Mapper — копия/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


KeplerMapper(verbose=3)
..Composing projection pipeline of length 1:
	Projections: knn_distance_5
	Distance matrices: False
	Scalers: MinMaxScaler()
..Projecting on data shaped (1962, 47)

..Projecting data using: knn_distance_5

..Scaling with: MinMaxScaler()

Mapping on data shaped (1962, 47) using lens shaped (1962, 2)

Minimal points in hypercube before clustering: 3
Creating 400 hypercubes.
Cube_0 is empty.

   > Found 3 clusters in hypercube 1.
   > Found 3 clusters in hypercube 2.
Cube_3 is empty.

Cube_4 is empty.

Cube_5 is empty.

Cube_6 is empty.

Cube_7 is empty.

   > Found 3 clusters in hypercube 8.
   > Found 3 clusters in hypercube 9.
   > Found 3 clusters in hypercube 10.
Cube_11 is empty.

Cube_12 is empty.

Cube_13 is empty.

Cube_14 is empty.

Cube_15 is empty.

   > Found 3 clusters in hypercube 16.
   > Found 3 clusters in hypercube 17.
   > Found 3 clusters in hypercube 18.
Cube_19 is empty.

Cube_20 is empty.

   > Found 3 clusters in hypercube 21.
   > Found 3 

In [2]:
app.run_server(debug=False, port=8052)